# Notebook 2: Online Feature Store Setup

Sets up the Postgres-backed Online Service, registers stream feature views with continuous aggregation for all 4 entity velocity groups, and benchmarks freshness and lookup latency.

---

### Architecture

```
FRAUD_TRANSACTIONS (source of truth)
      │
      └──► REST Ingest API ──► Online Feature Store (Postgres)
                                  ├── Stream FVs: CONTINUOUS aggregation (< 2s)
                                  └── Batch FV: Customer profile (daily)
```

### Feature Coverage

| Entity | Stream aggregations | Notes |
|---|---|---|
| Customer | count, sum, max, min, approx_count_distinct — 1h/6h/24h/48h/1wk | Primary card-testing signal |
| Merchant | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Merchant under attack |
| Wallet DPAN | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Compromised card |
| IP Address | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Bot farm |
| Customer Profile | Lifetime stats, account age | Batch online FV, daily refresh |

Derived ratio features (velocity ratios, concentration, deviation) are computed inline in the SPCS scoring container.

> **Preview note:** Requires `snowflake-ml-python >= 1.41`. Auth token is derived automatically from the active session when running inside Snowflake Notebooks.

In [ ]:
# ── Package version check ──────────────────────────────────────────────────
# IMPORTANT: If you just ran !pip install, you MUST restart the kernel before
# proceeding. Session → Restart Kernel → then re-run all cells from the top.
# pip install updates packages on disk but the running kernel keeps old modules
# in memory until a full restart.
#
# To avoid this: add snowflake-ml-python via the Packages picker at the top of
# the notebook (search "snowflake-ml-python", pin to >=1.41). That rebuilds the
# environment automatically and persists across sessions.
# ───────────────────────────────────────────────────────────────────────────

import importlib.metadata, sys

v = importlib.metadata.version('snowflake-ml-python')
major, minor = int(v.split('.')[0]), int(v.split('.')[1])
if (major, minor) < (1, 41):
    raise RuntimeError(
        f'snowflake-ml-python {v} is too old. Requires >= 1.41.\n'
        'Install via the Packages picker at the top of this notebook,\n'
        'then restart the kernel and re-run from the top.'
    )

# Verify the method actually exists in the running kernel (not just on disk)
from snowflake.ml.feature_store import FeatureStore as _FS
if not hasattr(_FS, 'create_online_service'):
    raise RuntimeError(
        f'create_online_service not found on FeatureStore (sdk={v}).\n'
        'Kernel is running an old cached version.\n'
        'Restart the kernel (Session → Restart Kernel) then re-run all cells.'
    )

print(f'snowflake-ml-python {v} — OK')
print(f'create_online_service available — OK')
print(f'Python: {sys.executable}')

In [ ]:
import time, os, numpy as np, requests, random, concurrent.futures
from datetime import datetime
from snowflake.snowpark.context import get_active_session

# Correct Preview API imports — requires snowflake-ml-python >= 1.41
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, Entity, CreationMode,
    OnlineConfig, OnlineStoreType, StreamSource, StreamConfig, Feature,
    FeatureGroup,
)
from snowflake.ml.feature_store.spec.enums import FeatureAggregationMethod
from snowflake.ml.feature_store import online_service as ofs_utils
from snowflake.ml.feature_store.online_service import OnlineServiceAccess
from snowflake.snowpark.types import (
    StructType, StructField, StringType, FloatType,
    TimestampType, TimestampTimeZone,
)

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA FEATURES').collect()
session.sql('USE WAREHOUSE FRAUD_DEMO_WH').collect()

fs = FeatureStore(
    session=session,
    database='FRAUD_DEMO_DEV',
    name='FEATURE_STORE',
    default_warehouse='FRAUD_DEMO_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
    # PUBLIC: nb02 runs in Snowflake Notebooks, NOT inside SPCS.
    # OnlineServiceAccess.INTERNAL forces SPCS-cluster-internal URLs which are
    # unreachable from Notebooks — all REST calls would fail with a connection error.
    # Use PUBLIC here; use INTERNAL only in nb04/nb06 which run inside the SPCS scoring container.
    online_service_access=OnlineServiceAccess.PUBLIC,
)

# Session token — no PAT env var needed when running inside Snowflake Notebooks
PAT = session.connection.rest.token or os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    raise RuntimeError('Could not obtain session token. If running locally set SNOWFLAKE_PAT.')
print(f'Auth token     : from active session ({PAT[:12]}...)')
print('Feature Store  : FRAUD_DEMO_DEV.FEATURE_STORE')

## 2.1 Create Online Service

Provisions a Postgres-backed serving layer co-located with your Snowflake account. Created once per Feature Store — takes 3-5 minutes on first run.

Exposes REST ingest and query endpoints (public, PrivateLink, or SPCS-internal). For production, configure PrivateLink to keep traffic off the public internet.

In [ ]:
print('Creating Online Service...\nFirst creation takes 3-5 minutes. Polling until RUNNING.\n')

try:
    fs.create_online_service('FRAUD_MLOPS', 'FRAUD_MLOPS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Online service already exists — continuing')
    else:
        raise

status = fs.get_online_service_status()
start  = time.time()
while status.status != 'RUNNING':
    print(f'  [{time.time()-start:.0f}s] {status.status} — waiting 30s...')
    time.sleep(30)
    status = fs.get_online_service_status()

print(f'Online Service RUNNING ({time.time()-start:.0f}s)')

# ── Resolve PUBLIC endpoint URLs via SQL function ───────────────────────────
# endpoint_url() auto-detects the environment. Snowflake Notebooks runs on SPCS
# infrastructure, so it returns SPCS-internal URLs (*.svc.spcs.internal) — but
# Notebooks is on a DIFFERENT compute pool namespace from the OFS service.
# The internal hostname is not resolvable from Notebooks' namespace.
#
# Fix: query SYSTEM$GET_FEATURE_STORE_ONLINE_SERVICE_STATUS directly to get
# all endpoint types, then explicitly pick the public URLs.
import json as _json

_raw = session.sql(
    "SELECT SYSTEM$GET_FEATURE_STORE_ONLINE_SERVICE_STATUS('FRAUD_DEMO_DEV.FEATURE_STORE')"
).collect()[0][0]
_svc = _json.loads(_raw)

# Print the full JSON so we can see the exact structure if parsing fails
print('\nRaw service status:')
print(_json.dumps(_svc, indent=2))

# Parse endpoint URLs — prefer public > private_link > internal
# The exact key names depend on the SDK version; we try common variants.
_endpoints = _svc.get('endpoints', _svc.get('Endpoints', {}))

def _pick_public(ep_name):
    """Return the public URL for the given endpoint name, trying common key variants."""
    # Try the endpoint by name first, then fall back to scanning all keys
    candidates = [
        _endpoints.get(ep_name, {}),
        _endpoints.get(ep_name.lower(), {}),
        _endpoints.get(ep_name.upper(), {}),
    ]
    for ep in candidates:
        if isinstance(ep, dict):
            url = ep.get('public') or ep.get('Public') or ep.get('publicUrl')
            if url:
                return url.rstrip('/')
            url = ep.get('privatelink') or ep.get('private_link') or ep.get('PrivateLink')
            if url:
                return url.rstrip('/')
        elif isinstance(ep, str):
            return ep.rstrip('/')
    # If the endpoint is a flat dict of name→url (not nested), scan for public-looking URLs
    for k, v in _endpoints.items():
        if ep_name.lower() in k.lower() and isinstance(v, str) and 'internal' not in v:
            return v.rstrip('/')
    raise RuntimeError(
        f'Could not find public URL for {ep_name!r}.\n'
        f'Full endpoints dict: {_json.dumps(_endpoints, indent=2)}'
    )

QUERY_URL  = _pick_public('query')
INGEST_URL = _pick_public('ingest')
HEADERS    = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}

print(f'\nUsing:')
print(f'  QUERY_URL  : {QUERY_URL}')
print(f'  INGEST_URL : {INGEST_URL}')
print(f'  Auth token : {PAT[:16]}...')

## 2.2 Register Entities

In [ ]:
# An Entity is a named primary key definition — it tells the Feature Store which column(s)
# to use when joining feature views to incoming scoring requests.
# join_keys must match the column names in both the stream schema and the scoring payload.
# We define 4 entities (one per velocity dimension): Customer, Merchant, DPAN, IP Address.
# Card Token velocity is derived at scoring time from DPAN features, so no separate entity needed.
customer_entity = Entity(name='FRAUD_CUSTOMER',    join_keys=['CUSTOMER_ID'])
merchant_entity = Entity(name='FRAUD_MERCHANT',    join_keys=['MERCHANT_ID'])
dpan_entity     = Entity(name='FRAUD_WALLET_DPAN', join_keys=['WALLET_DPAN'])
ip_entity       = Entity(name='FRAUD_IP_ADDRESS',  join_keys=['IP_ADDRESS'])

# Register each entity in the Feature Store (idempotent — safe to re-run).
# Entities are lightweight metadata objects; they don't store any data themselves.
for entity in [customer_entity, merchant_entity, dpan_entity, ip_entity]:
    try:
        fs.register_entity(entity)
        print(f'  Registered: {entity.name}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'  Exists: {entity.name}')
        else:
            raise

## 2.3 Register Stream Source

Defines the event schema for real-time transaction ingestion. Every transaction is sent here via REST to trigger continuous velocity feature updates.

In [ ]:
from snowflake.snowpark.types import DoubleType

# A StreamSource defines the schema of the event records that will be ingested into the
# Online Feature Store. Each time a transaction is scored, a record is posted to the
# INGEST endpoint — the OFS uses these records to maintain rolling window aggregates.
#
# Only the fields needed for feature computation are included here (not the full transaction).
# AMOUNT_USD: the transaction amount — used for sum/count/max velocity features.
# IS_GBR:     1.0 if merchant country = GBR, else 0.0 — precomputed to avoid string ops.
# EVENT_TS:   the transaction timestamp — defines the time axis for all rolling windows.
#             TimestampTimeZone.NTZ = no time zone (UTC assumed, consistent with TIMESTAMP_NTZ).
txn_stream = StreamSource(
    name='FRAUD_TXN_EVENTS',
    schema=StructType([
        StructField('CUSTOMER_ID', StringType()),
        StructField('MERCHANT_ID', StringType()),
        StructField('WALLET_DPAN', StringType()),
        StructField('IP_ADDRESS',  StringType()),
        StructField('AMOUNT_USD',  DoubleType()),
        StructField('IS_GBR',      DoubleType()),
        StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
    ]),
    desc='Real-time fraud transaction events',
)

# register_stream_source creates the underlying Snowflake stream object.
# Safe to re-run — catches 'already exists' errors gracefully.
try:
    fs.register_stream_source(txn_stream)
    print('Stream source registered: FRAUD_TXN_EVENTS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Stream source already exists — continuing')
    else:
        raise

## 2.4 Create Stream Feature Views (CONTINUOUS Aggregation)

`FeatureAggregationMethod.CONTINUOUS` maintains running aggregates updated on every ingest event — no refresh cycle, no warehouse, < 2s end-to-end freshness.

**Why this matters:** A card-testing burst of 5 transactions spanning 30 seconds is invisible to a 33-39s refresh cycle. With CONTINUOUS aggregation, velocity counts update after each transaction — the model detects the burst from transaction 2 onwards.

> **approx_count_distinct note:** Distinct-value features (distinct merchants, DPANs) use HyperLogLog (~6.5% RSE at default precision). For fraud velocity signals this is negligible. Use `precision=14` to reduce RSE to ~0.8% if needed.

In [ ]:
import pandas as pd
from datetime import datetime

online_cfg = OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES)

def identity_transform(df: pd.DataFrame) -> pd.DataFrame:
    """Pass events through unchanged — all aggregations handled by Feature.count/sum/max."""
    return df

backfill_schema = StructType([
    StructField('CUSTOMER_ID', StringType()),
    StructField('MERCHANT_ID', StringType()),
    StructField('WALLET_DPAN', StringType()),
    StructField('IP_ADDRESS',  StringType()),
    StructField('AMOUNT_USD',  DoubleType()),
    StructField('IS_GBR',      DoubleType()),
    StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
])
backfill_df = session.create_dataframe(
    [['DUMMY', 'DUMMY', 'DUMMY', '0.0.0.0', 0.0, 0.0, datetime(2020, 1, 1)]],
    schema=backfill_schema,
)

stream_cfg = StreamConfig(
    stream_source=txn_stream,
    transformation_fn=identity_transform,
    backfill_df=backfill_df,
)

# ── Customer velocity (primary card-testing signal) ──────────────────────────
# CONTINUOUS: features update on every REST ingest event, not on a batch schedule.
# refresh_freq / feature_granularity: '1 minute' tile size for the online store.
customer_fv = FeatureView(
    name='FRAUD_CUSTOMER_VELOCITY',
    entities=[customer_entity],
    stream_config=stream_cfg,
    timestamp_col='EVENT_TS',
    refresh_freq='1 minute',
    feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('PURCHASES_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('PURCHASES_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('PURCHASES_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('PURCHASES_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('PURCHASES_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '6h' ).alias('PURCHASES_AMT_L6H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('PURCHASES_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '48h').alias('PURCHASES_AMT_L48H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_MAX_L1H'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('PURCHASES_MAX_L24H'),
        Feature.max(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_MAX_L1WK'),
        Feature.min(   'AMOUNT_USD',   '24h').alias('PURCHASES_MIN_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '1h' ).alias('DISTINCT_MERCHANTS_L1H'),
        Feature.approx_count_distinct('MERCHANT_ID', '6h' ).alias('DISTINCT_MERCHANTS_L6H'),
        Feature.approx_count_distinct('MERCHANT_ID', '24h').alias('DISTINCT_MERCHANTS_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '7d' ).alias('DISTINCT_MERCHANTS_L1WK'),
        Feature.approx_count_distinct('WALLET_DPAN', '24h').alias('DISTINCT_DPANS_L24H'),
        Feature.approx_count_distinct('WALLET_DPAN', '7d' ).alias('DISTINCT_DPANS_L1WK'),
        Feature.count('IS_GBR', '24h').alias('PURCHASES_GBR_NUM_L24H'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Customer velocity — CONTINUOUS, < 2s freshness',
)
try:
    customer_fv = fs.register_feature_view(customer_fv, version='V1', block=True)
    print('  Registered: FRAUD_CUSTOMER_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        customer_fv = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
        print('  Already registered: FRAUD_CUSTOMER_VELOCITY')
    else:
        raise

In [ ]:
# ── Merchant velocity ────────────────────────────────────────────────────────
# Merchant-level features detect compromised merchants and bust-out fraud rings.
# Key signal: a sudden spike in MERCHANT_UNIQUE_CUST_L1H (many different customers
# at one merchant in 1 hour) can indicate a card-skimming or POS-compromise incident.
merchant_fv = FeatureView(
    name='FRAUD_MERCHANT_VELOCITY',
    entities=[merchant_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('MERCHANT_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('MERCHANT_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('MERCHANT_MAX_TXN_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('MERCHANT_UNIQUE_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('MERCHANT_UNIQUE_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('MERCHANT_UNIQUE_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Merchant velocity — CONTINUOUS',
)
try:
    merchant_fv = fs.register_feature_view(merchant_fv, version='V1', block=True)
    print('  Registered: FRAUD_MERCHANT_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        merchant_fv = fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1')
        print('  Already registered: FRAUD_MERCHANT_VELOCITY')
    else:
        raise

# ── Wallet DPAN velocity ──────────────────────────────────────────────────────
# DPAN = Device Primary Account Number (the tokenised card number provisioned to a wallet).
# Key signal: DPAN_DISTINCT_CUST_L24H > 1 means the same device token is being used by
# multiple customers — a strong indicator of DPAN cloning or account sharing fraud.
dpan_fv = FeatureView(
    name='FRAUD_DPAN_VELOCITY',
    entities=[dpan_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('DPAN_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('DPAN_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('DPAN_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('DPAN_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('DPAN_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('DPAN_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('DPAN_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='DPAN velocity — shared DPAN = fraud signal',
)
try:
    dpan_fv = fs.register_feature_view(dpan_fv, version='V1', block=True)
    print('  Registered: FRAUD_DPAN_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        dpan_fv = fs.get_feature_view('FRAUD_DPAN_VELOCITY', 'V1')
        print('  Already registered: FRAUD_DPAN_VELOCITY')
    else:
        raise

# ── IP Address velocity ───────────────────────────────────────────────────────
# IP-level features detect botnet traffic and proxy/VPN fraud rings.
# Key signal: IP_DISTINCT_CUST_L1H > threshold means many different customers
# are transacting from the same IP in a short window — typical of a bot farm or NAT proxy.
ip_fv = FeatureView(
    name='FRAUD_IP_VELOCITY',
    entities=[ip_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('IP_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('IP_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('IP_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('IP_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('IP_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('IP_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('IP_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('IP_DISTINCT_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('IP_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('IP_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='IP velocity — shared IP = bot farm signal',
)
try:
    ip_fv = fs.register_feature_view(ip_fv, version='V1', block=True)
    print('  Registered: FRAUD_IP_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        ip_fv = fs.get_feature_view('FRAUD_IP_VELOCITY', 'V1')
        print('  Already registered: FRAUD_IP_VELOCITY')
    else:
        raise
print('\nAll 4 entity stream feature views registered.')

## 2.5 Customer Profile Feature View (Batch, Daily Refresh)

Static lifetime features that change slowly. Served from a DT-backed batch online feature view refreshed daily.

In [ ]:
# The customer profile feature view is a BATCH feature view (not CONTINUOUS/stream-based).
# It reads from a pre-aggregated table (CUSTOMER_PROFILE) that is refreshed daily.
# Batch feature views are appropriate for slowly-changing attributes like:
#   - Lifetime transaction count and total spend
#   - Account age in days
#   - Average transaction amount (30-day rolling)
# These features don't need sub-second freshness — daily refresh is sufficient.

# CUSTOMER_PROFILE doesn't exist as a table — build it by joining DIM_CUSTOMERS
# (static attributes) with aggregates from FRAUD_TRANSACTIONS.
session.sql('''
    CREATE TABLE IF NOT EXISTS FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE AS
    SELECT
        c.CUSTOMER_ID,
        c.CUSTOMER_AGE,
        c.DAYS_SINCE_REGISTRATION,
        c.IS_HIGH_RISK,
        COALESCE(t.LIFETIME_TXN_COUNT, 0)    AS LIFETIME_TXN_COUNT,
        COALESCE(t.LIFETIME_TXN_TOTAL, 0.0)  AS LIFETIME_TXN_TOTAL,
        COALESCE(t.AVG_TXN_AMOUNT_30D, 0.0)  AS AVG_TXN_AMOUNT_30D
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c
    LEFT JOIN (
        SELECT
            CUSTOMER_ID,
            COUNT(*)                          AS LIFETIME_TXN_COUNT,
            SUM(PURCHASE_AMOUNT)              AS LIFETIME_TXN_TOTAL,
            AVG(CASE WHEN TRANSACTION_TS >= DATEADD(DAY, -30, CURRENT_TIMESTAMP())
                     THEN PURCHASE_AMOUNT END) AS AVG_TXN_AMOUNT_30D
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
        GROUP BY CUSTOMER_ID
    ) t ON c.CUSTOMER_ID = t.CUSTOMER_ID
''').collect()
print('Created: FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_src = session.table('FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_fv = FeatureView(
    name='FRAUD_CUSTOMER_PROFILE',
    entities=[customer_entity],
    feature_df=profile_src,
    refresh_freq='1 day',
    online_config=OnlineConfig(enable=True, target_lag='1h', store_type=OnlineStoreType.POSTGRES),
    desc='Customer profile: lifetime stats, account age. Batch online FV, daily refresh.',
)
profile_fv = profile_fv = try:
    profile_fv = fs.register_feature_view(profile_fv, version='V1', block=True)
    print('Registered: FRAUD_CUSTOMER_PROFILE (batch, daily refresh)')

## 2.6 Register Feature Group (Single-Call Scoring)

`FeatureGroup` bundles all 5 feature views into a single `read_feature_group()` call.
This replaces 4-5 concurrent REST calls with **one round-trip** to the online service.

All source FVs must use `store_type=OnlineStoreType.POSTGRES` — verified above.

In [ ]:
# FeatureGroup bundles all 5 feature views for single-call scoring.
# 'a single read_feature_group() call returns features from all sources in one round-trip'
# This reduces 5 concurrent REST calls -> 1 call on the hot path.
#
# Use fs.get_feature_view() to retrieve the *registered* versions from the server.
# FeatureGroup requires registered FV objects, not local unregistered ones.
# This also makes the cell idempotent: works whether FVs were registered this session or earlier.
fraud_fg = FeatureGroup(
    name='FRAUD_SCORING_FG',
    features=[
        fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1'),
        fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1'),
        fs.get_feature_view('FRAUD_DPAN_VELOCITY',     'V1'),
        fs.get_feature_view('FRAUD_IP_VELOCITY',       'V1'),
        fs.get_feature_view('FRAUD_CUSTOMER_PROFILE',  'V1'),
    ],
    auto_prefix=False,
    desc='All fraud scoring features: 4 velocity views (CONTINUOUS, <2s) + customer profile',
)

try:
    registered_fg = fs.register_feature_group(fraud_fg, version='V1')
    print(f'Registered FeatureGroup: {registered_fg.name} V1')
except Exception as e:
    if 'already exists' in str(e).lower():
        registered_fg = fs.get_feature_group('FRAUD_SCORING_FG', 'V1')
        print(f'FeatureGroup already exists: {registered_fg.name} V1')
    else:
        raise

print()
print('Scoring path: fs.read_feature_group(fraud_fg, keys=[[cust_id, merch_id, dpan, ip]])')
print('  -> ONE round-trip -> all velocity + profile features')
print('  -> XGBoost.predict() (~5ms)')


## 2.6 Freshness Benchmark (RUN LIVE)

Measures time from REST ingest to feature visible at query time. Target: < 2 seconds.

In [ ]:
# Freshness benchmark: measures how quickly an ingested event propagates to ALL 4 feature views.
# A single transaction fans out to 4 independent entity pipelines (customer, merchant, DPAN, IP).
# The "time to score" = max latency across all 4, since scoring can't proceed until all are fresh.
#
# NOTE: This cell requires nb02 cells 1-4 to have been run successfully first so that
# QUERY_URL, INGEST_URL, and HEADERS are defined.

from datetime import datetime, timezone

def _ofs_query(fv_name, key_col, entity_val, features):
    """Single feature view query; raises on HTTP error with full response body."""
    r = requests.post(
        f'{QUERY_URL}/api/v1/query',
        headers=HEADERS,
        json={
            'name': fv_name, 'version': 'V1', 'object_type': 'feature_view',
            'request_rows': [{'entity': {key_col: entity_val}}],
            'features': features,
            'metadata_options': {'include_serving_status': True},
        },
        timeout=10,
    )
    if r.status_code not in (200, 207):
        raise RuntimeError(
            f'Query {fv_name} failed: HTTP {r.status_code}\n'
            f'URL: {QUERY_URL}/api/v1/query\n'
            f'Response: {r.text[:500]}'
        )
    return r.json()

print('='*60)
print('FRESHNESS BENCHMARK')
print('='*60)
print(f'QUERY_URL  : {QUERY_URL}')
print(f'INGEST_URL : {INGEST_URL}')
print(f'Auth token : {PAT[:16]}...')
print()

# The 4 feature views and their entity key columns
FV_ENTITIES = [
    ('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID'),
    ('FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID'),
    ('FRAUD_DPAN_VELOCITY',     'WALLET_DPAN'),
    ('FRAUD_IP_VELOCITY',       'IP_ADDRESS'),
]
FEATURE_PER_FV = {
    'FRAUD_CUSTOMER_VELOCITY': 'PURCHASES_NUM_L1H',
    'FRAUD_MERCHANT_VELOCITY': 'MERCHANT_TXN_NUM_L1H',
    'FRAUD_DPAN_VELOCITY':     'DPAN_TXN_NUM_L1H',
    'FRAUD_IP_VELOCITY':       'IP_TXN_NUM_L1H',
}

# Fresh unique keys — guaranteed no prior history in any window
test_cust  = f'BENCH_CUST_{random.randint(900000, 999999)}'
test_merch = f'BENCH_MERCH_{random.randint(900000, 999999)}'
test_dpan  = f'BENCH_DPAN_{random.randint(900000, 999999)}'
test_ip    = f'BENCH_IP_{random.randint(900000, 999999)}'
entity_keys = {
    'CUSTOMER_ID': test_cust,
    'MERCHANT_ID': test_merch,
    'WALLET_DPAN': test_dpan,
    'IP_ADDRESS':  test_ip,
}
print(f'Test keys: {test_cust} | {test_merch} | {test_dpan} | {test_ip}')

# ── Step 1: Auth + baseline smoke test ─────────────────────────────────────
print('\n--- Step 1: Baseline queries (expect null/0 for unseen keys) ---')
for fv_name, key_col in FV_ENTITIES:
    resp = _ofs_query(fv_name, key_col, entity_keys[key_col], [FEATURE_PER_FV[fv_name]])
    results = resp.get('results', [])
    val = (results[0].get('features', [None])[0]) if results else None
    status = (results[0].get('status', ['?'])[0]) if results else '?'
    print(f'  {fv_name}: {FEATURE_PER_FV[fv_name]} = {val}  ({status})')
print('  Auth OK — all baseline queries succeeded')

# ── Step 2: Ingest single event ─────────────────────────────────────────────
print('\n--- Step 2: Ingest event ---')
ingest_ts  = datetime.now(timezone.utc)
test_amount = round(random.uniform(10, 500), 2)
ir = requests.post(
    f'{INGEST_URL}/api/v1/ingest',
    headers=HEADERS,
    json={
        'dry_run': False,
        'include_diagnostics': True,
        'records': {'FRAUD_TXN_EVENTS': [{
            'CUSTOMER_ID': test_cust,
            'MERCHANT_ID': test_merch,
            'WALLET_DPAN': test_dpan,
            'IP_ADDRESS':  test_ip,
            'AMOUNT_USD':  test_amount,
            'IS_GBR':      1.0,
            'EVENT_TS':    ingest_ts.strftime('%Y-%m-%dT%H:%M:%SZ'),
        }]},
    },
    timeout=10,
)
print(f'  HTTP {ir.status_code}')
if ir.status_code not in (200, 207):
    raise RuntimeError(
        f'Ingest failed: HTTP {ir.status_code}\n'
        f'URL: {INGEST_URL}/api/v1/ingest\n'
        f'Response: {ir.text[:1000]}'
    )

# Print which feature views accepted the event
ingest_resp = ir.json()
src = ingest_resp.get('sources', {}).get('FRAUD_TXN_EVENTS', {})
fv_results  = src.get('feature_views', {}).get('results', [])
rec_failed  = src.get('records', {}).get('failed', 0)
if rec_failed:
    raise RuntimeError(f'Record validation failed: {src.get("records", {})}')
for fv_r in fv_results:
    print(f'  FV {fv_r["name"]}/{fv_r["version"]}: {fv_r["status"]}')
if not fv_results:
    print(f'  (raw ingest response: {ingest_resp})')

start = time.time()
print(f'  Ingest accepted — polling started')

# ── Step 3: Poll all 4 FVs until count feature > 0 ─────────────────────────
print('\n--- Step 3: Polling (250ms intervals, 30s timeout) ---')
latencies = {}
pending   = set(fv for fv, _ in FV_ENTITIES)

for i in range(120):   # 120 × 250ms = 30s
    time.sleep(0.25)
    for fv_name, key_col in FV_ENTITIES:
        if fv_name not in pending:
            continue
        try:
            resp    = _ofs_query(fv_name, key_col, entity_keys[key_col], [FEATURE_PER_FV[fv_name]])
            results = resp.get('results', [])
            val     = (results[0].get('features', [None])[0]) if results else None
            if val is not None and val > 0:
                ms = (time.time() - start) * 1000
                latencies[fv_name] = ms
                pending.discard(fv_name)
                print(f'  {fv_name}: {ms:.0f}ms  ({FEATURE_PER_FV[fv_name]} = {val})')
        except Exception as exc:
            print(f'  ERROR polling {fv_name}: {exc}')
            pending.discard(fv_name)   # stop retrying this FV
    if not pending:
        break
    if i % 8 == 0 and i > 0:
        print(f'  [{i*0.25:.1f}s] waiting: {", ".join(sorted(pending))}')

# ── Step 4: Summary ─────────────────────────────────────────────────────────
print('\n--- Results ---')
if pending:
    print(f'  TIMEOUT (>30s): {", ".join(sorted(pending))}')
if latencies:
    for fv, ms in sorted(latencies.items(), key=lambda x: x[1]):
        print(f'  {fv}: {ms:.0f}ms')
    worst = max(latencies.values())
    print(f'\n  Fastest (single pipeline) : {min(latencies.values()):.0f}ms')
    print(f'  Slowest (time-to-score)   : {worst:.0f}ms')
    verdict = "PASS ✓" if worst < 2000 else "FAIL ✗"
    print(f'  Verdict: {worst:.0f}ms — {verdict} (target < 2000ms)')

## 2.7 Latency Benchmark

10 warmup + 200 measured requests. Single-entity and 4-entity concurrent (per-transaction) lookups using real entity keys.

In [ ]:
print('='*60)
print('QUERY LATENCY BENCHMARK  (public URL from Notebooks)')
print('='*60)
print('NOTE: These numbers use the public OFS URL (Notebooks is not in SPCS).')
print('For production-equivalent internal-URL latency, see nb06_latency_proof.ipynb.')
print('''
What this measures:
  The HTTP round-trip time to READ features from the Online Feature Store.
  No data is ingested — we are purely measuring how fast the Postgres-backed
  online store responds to point lookups by entity key.

How it differs from the Freshness Benchmark:
  • Freshness = time for a NEW event to propagate through the CONTINUOUS
    pipeline and become queryable (write path → aggregation → read path).
    It also queries all 4 feature views, but it measures the WRITE PIPELINE
    delay — how long until freshly ingested data is visible.
  • Latency = time to READ already-computed features (read path only).
    No ingestion occurs. Features already exist in the online store.
    This isolates the Postgres lookup + HTTP overhead from pipeline processing.

  In production, total scoring time = freshness + latency:
    - Freshness tells you how stale features might be at query time.
    - Latency tells you how long the scoring service blocks waiting for
      the feature vector response.

Benchmark 1: Single-entity lookup (customer only)
  → Measures raw OFS query overhead for one feature view.

Benchmark 2: 4-entity concurrent lookup (customer + merchant + DPAN + IP)
  → Simulates a real scoring call: 4 lookups fired in parallel.
  → Reports wall-clock time = max(4 individual latencies).
  → This is what the fraud model waits for per transaction.
''')

N_WARMUP, N_REQUESTS = 10, 200

# Sample 250 real entity keys from the transactions table to use as test inputs.
# Using real keys (rather than random strings) ensures cache behaviour is realistic.
samples = session.sql('SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (250 ROWS)').collect()
print(f'Sampled {len(samples)} entity keys\n')

def qlat(name, key_col, key_val):
    """Issue a single feature view query and return the round-trip latency in ms."""
    t0 = time.perf_counter()
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: key_val}}],
    }, timeout=10)
    r.raise_for_status()
    return (time.perf_counter() - t0) * 1000

# Benchmark 1: single entity
print('='*55); print('BENCHMARK 1: Single-entity (customer)'); print('='*55)
print('  Metric: HTTP round-trip for one feature view query')
lat_s = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    lat = qlat('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID'])
    if i >= N_WARMUP: lat_s.append(lat)
    if i == N_WARMUP - 1: print('  Warmup complete (first 10 requests discarded)')
arr_s = np.array(lat_s)
print(f'  n={N_REQUESTS} requests')
print(f'  p50: {np.percentile(arr_s,50):.1f}ms  p95: {np.percentile(arr_s,95):.1f}ms  p99: {np.percentile(arr_s,99):.1f}ms')

# Benchmark 2: 4-entity concurrent
print(f'\n{"="*55}'); print('BENCHMARK 2: 4-entity concurrent (per-transaction)'); print('='*55)
print('  Metric: wall-clock time for 4 parallel feature view queries')
def q_all(row):
    """Fire 4 entity lookups in parallel; return the wall-clock time (= slowest of the 4)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        fs_ = [ex.submit(qlat, 'FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID']),
               ex.submit(qlat, 'FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', row['MERCHANT_ID']),
               ex.submit(qlat, 'FRAUD_DPAN_VELOCITY',     'WALLET_DPAN',  row['WALLET_DPAN']),
               ex.submit(qlat, 'FRAUD_IP_VELOCITY',       'IP_ADDRESS',   row['IP_ADDRESS'])]
    return max(f.result() for f in fs_)

lat_m = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    wall = q_all(row)
    if i >= N_WARMUP: lat_m.append(wall)
    if i == N_WARMUP - 1: print('  Warmup complete (first 10 requests discarded)')
arr_m = np.array(lat_m)
print(f'  n={N_REQUESTS} requests')
print(f'  p50: {np.percentile(arr_m,50):.1f}ms  p95: {np.percentile(arr_m,95):.1f}ms  p99: {np.percentile(arr_m,99):.1f}ms')
print(f'\n  → This is the per-transaction feature fetch overhead in production.')

---
## Summary

| Component | Freshness | Method |
|---|---|---|
| FRAUD_CUSTOMER_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_MERCHANT_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_DPAN_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_IP_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_CUSTOMER_PROFILE | Daily | Batch online FV |

**Online Service pricing** depends on entity cardinality, feature view count, ingest rate, and time-window depth. No 24/7 DT warehouse required.

**Next:** NB03 — train XGBoost from the transactions table.